# Data Quality Assessment

## Overview

This notebook evaluates the quality, integrity, completeness, and consistency of the Movie Analytics database.

The objective is to identify data quality issues that could affect downstream SQL transformations, analytics marts, exploratory analysis, and business reporting.

The assessment will be performed using SQL through SQLAlchemy, with Pandas used to retrieve, organize, and analyze the results.

## Data Quality Dimensions

The assessment will focus on the following dimensions:

- **Completeness** – Are required fields populated?
- **Uniqueness** – Are records and key fields unique where expected?
- **Validity** – Do values conform to expected formats and business rules?
- **Consistency** – Are values represented consistently?
- **Accuracy** – Are values reasonable within their expected context?
- **Integrity** – Are relationships between tables correctly maintained?

## Quality Checks

The following checks will be performed:

1. Table-level record counts
2. Missing value assessment
3. Primary key uniqueness
4. Duplicate record detection
5. Foreign key integrity
6. Orphan record detection
7. Empty string detection
8. Invalid numerical values
9. Invalid date values
10. Categorical consistency
11. Many-to-many relationship validation
12. Overall data quality summary

## Expected Outcome

The assessment will identify data quality issues, quantify their impact, and provide recommendations for handling them before the data is used to build analytical models and marts.

> **Key Principle:** Data should be validated before it is transformed or used to generate business insights.

In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Importing the required libraries
import pandas as pd
from  sqlalchemy import create_engine
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
# Creating a database engine
db_path = Path.home() / "Documents" / "Movie DB SQL Analytics" / "Database" / "TMDB-a-4006.db"

engine = create_engine(f"sqlite:///{db_path}")

print("Database Connection established successfully!")

Database Connection established successfully!


## 1. Database Tables

Before performing individual quality checks, we will retrieve the complete list of tables available in the database.

This establishes the scope of the data quality assessment.

In [7]:
tables = pd.read_sql("""
SELECT name AS Table_Name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", engine)

tables

,Table_Name
0,actors
1,casts
2,genremap
3,genres
4,keywordmap
5,keywords
6,languagemap
7,languages
8,movies
9,oscars


## 2. Record Count Validation

Record counts provide a baseline for each table.

These counts will be useful when validating joins and transformations later in the project. Unexpected changes in record counts can indicate duplicate records, incorrect joins, or unintended row multiplication.

In [8]:
table_names = [
    "actors",
    "casts",
    "oscars",
    "movies",
    "keywords",
    "languages",
    "genres",
    "genremap",
    "keywordmap",
    "languagemap",
    "productioncompanymap",
    "productioncountrymap",
    "productioncountries",
    "productioncompanies"
]

In [9]:
row_counts = []

for table in table_names:
    query = f"""
    SELECT
        '{table}' AS Table_Name,
        COUNT(*) AS Row_Count
    FROM {table};
    """
    
    row_counts.append(pd.read_sql(query, engine))

row_counts = pd.concat(row_counts, ignore_index=True)

row_counts

,Table_Name,Row_Count
0,actors,54588
1,casts,106257
2,oscars,9964
3,movies,4803
4,keywords,9813
5,languages,87
6,genres,20
7,genremap,12160
8,keywordmap,36194
9,languagemap,6937


## 3. Missing Value Assessment

Missing values can affect calculations, joins, aggregations, and business reporting.

The assessment will identify the number and percentage of NULL values in each column.

Missing values will not automatically be replaced. Their treatment will depend on the business meaning of each field.

In [10]:
def missing_value_profile(table_name):
    query = f"SELECT * FROM {table_name} LIMIT 0"
    columns = pd.read_sql(query, engine).columns.tolist()
    
    results = []

    for column in columns:
        query = f"""
        SELECT
            '{table_name}' AS Table_Name,
            '{column}' AS Column_Name,
            COUNT(*) AS Total_Rows,
            SUM(CASE WHEN "{column}" IS NULL THEN 1 ELSE 0 END) AS Missing_Count
        FROM {table_name};
        """
        
        result = pd.read_sql(query, engine)
        result["Missing_Percentage"] = (
            result["Missing_Count"] /
            result["Total_Rows"] * 100
        )
        
        results.append(result)

    return pd.concat(results, ignore_index=True)

In [11]:
missing_results = []

for table in table_names:
    missing_results.append(
        missing_value_profile(table)
    )

missing_profile = pd.concat(
    missing_results,
    ignore_index=True
)

missing_profile.sort_values(
    "Missing_Percentage",
    ascending=False
).head(20)

,Table_Name,Column_Name,Total_Rows,Missing_Count,Missing_Percentage
8,oscars,winner,9964,7643,76.706142
15,movies,homepage,4803,3091,64.355611
29,languages,language_name,87,26,29.885057
23,movies,tagline,4803,844,17.572351
10,oscars,film,9964,333,3.342031
18,movies,overview,4803,3,0.062461
21,movies,runtime,4803,2,0.041641
13,movies,release_date,4803,1,0.020820
39,productioncompanymap,production_company_id,13677,0,0.000000
38,productioncompanymap,movie_id,13677,0,0.000000


## 4. Primary Key Integrity

Primary keys should uniquely identify records within their respective entity tables.

Duplicate primary keys can indicate data integrity problems.

The assessment will identify duplicate values in primary key fields.

In [15]:
def check_duplicates(table, columns):
    
    column_list = ", ".join(columns)
    
    query = f"""
    SELECT
        {column_list},
        COUNT(*) AS Record_Count
    FROM {table}
    GROUP BY {column_list}
    HAVING COUNT(*) > 1
    ORDER BY Record_Count DESC;
    """
    
    return pd.read_sql(query, engine)

In [16]:
check_duplicates("actors", ["actor_id"])

,actor_id,Record_Count


In [17]:
check_duplicates("movies", ["movie_id"])

,movie_id,Record_Count


In [18]:
check_duplicates("genres", ["genre_id"])

,genre_id,Record_Count


In [19]:
check_duplicates("keywords", ["keyword_id"])

,keyword_id,Record_Count


In [20]:
check_duplicates("languages", ["ISO_639_1"])

,iso_639_1,Record_Count


In [21]:
check_duplicates("productioncountries", ["ISO_3166_1"])

,iso_3166_1,Record_Count


In [22]:
check_duplicates("productioncompanies", ["production_company_id"])

,production_company_id,Record_Count


In [23]:
check_duplicates("casts", ["movie_id", "actor_id"])

,movie_id,actor_id,Record_Count
0,182291,567620,5
1,18360,85757,3
2,18360,130565,3
3,18360,130566,3
4,38167,1456290,3
...,...,...,...
160,329833,85757,2
161,333355,31162,2
162,333355,39214,2
163,339984,156100,2


In [24]:
check_duplicates("genremap", ["movie_id", "genre_id"])

,movie_id,genre_id,Record_Count


In [25]:
check_duplicates("keywordmap", ["movie_id", "keyword_id"])

,movie_id,keyword_id,Record_Count


In [26]:
check_duplicates("languagemap", ["movie_id", "ISO_639_1"])

,movie_id,iso_639_1,Record_Count


In [27]:
check_duplicates(
    "productioncompanymap",
    ["movie_id", "production_company_id"]
)

,movie_id,production_company_id,Record_Count


In [28]:
check_duplicates(
    "productioncountrymap",
    ["movie_id", "ISO_3166_1"]
)

,movie_id,iso_3166_1,Record_Count


## 6. Foreign Key Integrity

Foreign key validation determines whether records in child tables have corresponding records in their parent tables.

The objective is to identify orphan records that reference entities that do not exist.

Examples include:

- A cast record referencing a non-existent movie.
- A cast record referencing a non-existent actor.
- A genre mapping referencing a non-existent genre.
- A language mapping referencing a non-existent language.

Orphan records can cause information loss or unexpected results when INNER JOIN operations are performed.

In [29]:
pd.read_sql("""
SELECT COUNT(*) AS Orphan_Cast_Records
FROM casts c
LEFT JOIN movies m
    ON c.movie_id = m.movie_id
WHERE m.movie_id IS NULL;
""", engine)

,Orphan_Cast_Records
0,0


In [30]:
pd.read_sql("""
SELECT COUNT(*) AS Orphan_Cast_Records
FROM casts c
LEFT JOIN actors a
    ON c.actor_id = a.actor_id
WHERE a.actor_id IS NULL;
""", engine)

,Orphan_Cast_Records
0,0


## 7. Mapping Table Integrity

The database contains several bridge tables that represent many-to-many relationships.

These include:

- `genremap`
- `keywordmap`
- `languagemap`
- `productioncompanymap`
- `productioncountrymap`

These tables will be validated against both sides of their relationships.

The objective is to identify mappings that reference missing movies or missing dimension records.

In [32]:
orphan_checks = pd.DataFrame({
    "Relationship": [
        "casts → movies",
        "casts → actors",
        "genremap → movies",
        "genremap → genres",
        "keywordmap → movies",
        "keywordmap → keywords",
        "languagemap → movies",
        "languagemap → languages",
        "productioncompanymap → movies",
        "productioncompanymap → productioncompanies",
        "productioncountrymap → movies",
        "productioncountrymap → productioncountries",
        "oscars → movies"
    ],

    "Orphan_Count": [
        pd.read_sql("""
            SELECT COUNT(*)
            FROM casts c
            LEFT JOIN movies m
                ON c.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM casts c
            LEFT JOIN actors a
                ON c.actor_id = a.actor_id
            WHERE a.actor_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM genremap gm
            LEFT JOIN movies m
                ON gm.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM genremap gm
            LEFT JOIN genres g
                ON gm.genre_id = g.genre_id
            WHERE g.genre_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM keywordmap km
            LEFT JOIN movies m
                ON km.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM keywordmap km
            LEFT JOIN keywords k
                ON km.keyword_id = k.keyword_id
            WHERE k.keyword_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM languagemap lm
            LEFT JOIN movies m
                ON lm.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM languagemap lm
            LEFT JOIN languages l
                ON lm.ISO_639_1 = l.ISO_639_1
            WHERE l.ISO_639_1 IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM productioncompanymap pcm
            LEFT JOIN movies m
                ON pcm.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM productioncompanymap pcm
            LEFT JOIN productioncompanies pc
                ON pcm.production_company_id = pc.production_company_id
            WHERE pc.production_company_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM productioncountrymap pcom
            LEFT JOIN movies m
                ON pcom.movie_id = m.movie_id
            WHERE m.movie_id IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM productioncountrymap pcom
            LEFT JOIN productioncountries pc
                ON pcom.ISO_3166_1 = pc.ISO_3166_1
            WHERE pc.ISO_3166_1 IS NULL;
        """, engine).iloc[0, 0],

        pd.read_sql("""
            SELECT COUNT(*)
            FROM oscars o
            LEFT JOIN movies m
                ON TRIM(LOWER(o.film)) = TRIM(LOWER(m.title))
            WHERE m.title IS NULL;
        """, engine).iloc[0, 0]
    ]
})

orphan_checks

,Relationship,Orphan_Count
0,casts → movies,0
1,casts → actors,0
2,genremap → movies,0
3,genremap → genres,0
4,keywordmap → movies,0
5,keywordmap → keywords,0
6,languagemap → movies,0
7,languagemap → languages,0
8,productioncompanymap → movies,0
9,productioncompanymap → productioncompanies,0


In [33]:
orphan_checks["Status"] = orphan_checks["Orphan_Count"].apply(
    lambda x: "PASS" if x == 0 else "FAIL"
)

orphan_checks

,Relationship,Orphan_Count,Status
0,casts → movies,0,PASS
1,casts → actors,0,PASS
2,genremap → movies,0,PASS
3,genremap → genres,0,PASS
4,keywordmap → movies,0,PASS
5,keywordmap → keywords,0,PASS
6,languagemap → movies,0,PASS
7,languagemap → languages,0,PASS
8,productioncompanymap → movies,0,PASS
9,productioncompanymap → productioncompanies,0,PASS


## Empty String Assessment

Empty strings are values containing no characters (`''`) or only whitespace characters.

Unlike NULL values, empty strings are stored values and therefore require a separate validation check.

This assessment scans all text columns across the database to identify fields containing empty or whitespace-only values.

The results will help determine whether missing information is represented inconsistently as NULL, empty strings, or whitespace.

In [35]:
# Get all tables
tables = pd.read_sql("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", engine)

empty_string_results = []

for table in tables["table_name"]:
    
    # Get column information
    columns = pd.read_sql(
        f'PRAGMA table_info("{table}")',
        engine
    )
    
    # Select text columns
    text_columns = columns[
        columns["type"].str.upper().str.contains(
            "CHAR|TEXT|CLOB|VARCHAR"
        )
    ]["name"].tolist()
    
    for column in text_columns:
        
        query = f"""
        SELECT
            '{table}' AS Table_Name,
            '{column}' AS Column_Name,
            COUNT(*) AS Empty_String_Count
        FROM "{table}"
        WHERE TRIM("{column}") = '';
        """
        
        result = pd.read_sql(query, engine)
        
        if result["Empty_String_Count"].iloc[0] > 0:
            empty_string_results.append(result)

if empty_string_results:
    empty_string_profile = pd.concat(
        empty_string_results,
        ignore_index=True
    )
else:
    empty_string_profile = pd.DataFrame(
        columns=[
            "Table_Name",
            "Column_Name",
            "Empty_String_Count"
        ]
    )

empty_string_profile

,Table_Name,Column_Name,Empty_String_Count
0,casts,characters,683
1,movies,overview,1


In [36]:
empty_string_results = []

for table in tables["table_name"]:
    
    columns = pd.read_sql(
        f'PRAGMA table_info("{table}")',
        engine
    )
    
    text_columns = columns[
        columns["type"].str.upper().str.contains(
            "CHAR|TEXT|CLOB|VARCHAR"
        )
    ]["name"].tolist()
    
    for column in text_columns:
        
        query = f"""
        SELECT
            '{table}' AS Table_Name,
            '{column}' AS Column_Name,
            COUNT(*) AS Total_Rows,
            SUM(
                CASE 
                    WHEN "{column}" IS NULL THEN 1
                    ELSE 0
                END
            ) AS Null_Count,
            SUM(
                CASE
                    WHEN TRIM("{column}") = '' THEN 1
                    ELSE 0
                END
            ) AS Empty_String_Count
        FROM "{table}";
        """
        
        empty_string_results.append(
            pd.read_sql(query, engine)
        )

empty_string_profile = pd.concat(
    empty_string_results,
    ignore_index=True
)

empty_string_profile

,Table_Name,Column_Name,Total_Rows,Null_Count,Empty_String_Count
0,actors,actor_name,54588,0,0
1,casts,characters,106257,0,683
2,genres,genre_name,20,0,0
3,keywords,keyword_name,9813,0,0
4,languagemap,iso_639_1,6937,0,0
5,languages,iso_639_1,87,0,0
6,languages,language_name,87,26,0
7,movies,title,4803,0,0
8,movies,homepage,4803,3091,0
9,movies,original_language,4803,0,0


## Numerical Data Validity Assessment

Numerical columns are examined for values that may violate expected data constraints.

The assessment identifies:

- Negative values
- Minimum values
- Maximum values
- Zero values
- Potentially invalid ranges

The analysis is performed at column level because different numerical attributes have different valid ranges.

A value is not automatically classified as invalid simply because it is negative or zero. The result must be interpreted according to the business meaning of the column.

In [37]:
numeric_columns = []

for table in tables["table_name"]:
    
    columns = pd.read_sql(
        f'PRAGMA table_info("{table}")',
        engine
    )
    
    numeric_types = columns[
        columns["type"].str.upper().str.contains(
            "INT|REAL|FLOAT|DOUBLE|DECIMAL|NUMERIC"
        )
    ]
    
    for _, row in numeric_types.iterrows():
        numeric_columns.append({
            "Table_Name": table,
            "Column_Name": row["name"],
            "Data_Type": row["type"]
        })

numeric_columns = pd.DataFrame(numeric_columns)

numeric_columns

,Table_Name,Column_Name,Data_Type
0,actors,actor_id,INTEGER
1,actors,gender,INTEGER
2,casts,movie_id,INTEGER
3,casts,actor_id,INTEGER
4,genremap,movie_id,INTEGER
5,genremap,genre_id,INTEGER
6,genres,genre_id,INTEGER
7,keywordmap,movie_id,INTEGER
8,keywordmap,keyword_id,INTEGER
9,keywords,keyword_id,INTEGER


In [38]:
numerical_profile = []

for _, row in numeric_columns.iterrows():
    
    table = row["Table_Name"]
    column = row["Column_Name"]
    
    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Total_Rows,
        COUNT("{column}") AS Non_Null_Count,
        MIN("{column}") AS Minimum_Value,
        MAX("{column}") AS Maximum_Value,
        AVG("{column}") AS Average_Value
    FROM "{table}";
    """
    
    numerical_profile.append(
        pd.read_sql(query, engine)
    )

numerical_profile = pd.concat(
    numerical_profile,
    ignore_index=True
)

numerical_profile

,Table_Name,Column_Name,Total_Rows,Non_Null_Count,Minimum_Value,Maximum_Value,Average_Value
0,actors,actor_id,54588,54588,1,1893207,725024.789331
1,actors,gender,54588,54588,0,2,0.792647
2,casts,movie_id,106257,106257,5,459488,56237.55713
3,casts,actor_id,106257,106257,1,1893207,453207.109565
4,genremap,movie_id,12160,12160,5,459488,50812.516694
5,genremap,genre_id,12160,12160,12,10770,1906.939638
6,genres,genre_id,20,20,12,10770,3758.7
7,keywordmap,movie_id,36194,36194,5,459488,40312.524148
8,keywordmap,keyword_id,36194,36194,30,238222,48872.688595
9,keywords,keyword_id,9813,9813,30,238222,93314.804953


In [39]:
negative_value_results = []

for _, row in numeric_columns.iterrows():
    
    table = row["Table_Name"]
    column = row["Column_Name"]
    
    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Negative_Value_Count
    FROM "{table}"
    WHERE "{column}" < 0;
    """
    
    result = pd.read_sql(query, engine)
    
    if result["Negative_Value_Count"].iloc[0] > 0:
        negative_value_results.append(result)

if negative_value_results:
    negative_value_profile = pd.concat(
        negative_value_results,
        ignore_index=True
    )
else:
    negative_value_profile = pd.DataFrame(
        columns=[
            "Table_Name",
            "Column_Name",
            "Negative_Value_Count"
        ]
    )

negative_value_profile

,Table_Name,Column_Name,Negative_Value_Count


## Zero Value Assessment

Zero values are assessed separately from NULL and missing values.

A zero value may be:

- A legitimate measurement.
- A default value.
- An indication that information was unavailable.
- An invalid value depending on the business context.

Therefore, zero values will be identified but not automatically replaced or classified as errors.

Particular attention will be given to financial fields such as budget and revenue because zero values can affect calculations such as ROI.

In [41]:
zero_value_results = []

for _, row in numeric_columns.iterrows():
    
    table = row["Table_Name"]
    column = row["Column_Name"]
    
    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Total_Rows,
        
        COALESCE(
            SUM(
                CASE
                    WHEN "{column}" = 0 THEN 1
                    ELSE 0
                END
            ), 0
        ) AS Zero_Value_Count
        
    FROM "{table}";
    """
    
    result = pd.read_sql(query, engine)
    
    # Make sure the value is numeric
    zero_count = result["Zero_Value_Count"].iloc[0]
    
    if pd.notna(zero_count) and zero_count > 0:
        zero_value_results.append(result)

if zero_value_results:
    zero_value_profile = pd.concat(
        zero_value_results,
        ignore_index=True
    )
else:
    zero_value_profile = pd.DataFrame(
        columns=[
            "Table_Name",
            "Column_Name",
            "Total_Rows",
            "Zero_Value_Count"
        ]
    )

zero_value_profile

,Table_Name,Column_Name,Total_Rows,Zero_Value_Count
0,actors,gender,54588,27725
1,movies,budget,4803,1037
2,movies,popularity,4803,1
3,movies,revenue,4803,1427
4,movies,runtime,4803,35
5,movies,vote_average,4803,63
6,movies,vote_count,4803,62


In [43]:
result["Zero_Value_Count"] = (
    result["Zero_Value_Count"]
    .fillna(0)
    .astype(int)
)

if result["Zero_Value_Count"].iloc[0] > 0:
    zero_value_results.append(result)

In [44]:
negative_value_profile["Negative_Percentage"] = (
    negative_value_profile["Negative_Value_Count"]
    / numerical_profile.set_index(
        ["Table_Name", "Column_Name"]
    )
    .loc[
        list(
            zip(
                negative_value_profile["Table_Name"],
                negative_value_profile["Column_Name"]
            )
        ),
        "Total_Rows"
    ]
    .values
) * 100

In [45]:
numerical_quality = []

for _, row in numeric_columns.iterrows():
    
    table = row["Table_Name"]
    column = row["Column_Name"]
    
    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        
        COUNT(*) AS Total_Rows,
        
        COUNT("{column}") AS Non_Null_Count,
        
        SUM(
            CASE
                WHEN "{column}" IS NULL THEN 1
                ELSE 0
            END
        ) AS Null_Count,
        
        SUM(
            CASE
                WHEN "{column}" = 0 THEN 1
                ELSE 0
            END
        ) AS Zero_Count,
        
        SUM(
            CASE
                WHEN "{column}" < 0 THEN 1
                ELSE 0
            END
        ) AS Negative_Count,
        
        MIN("{column}") AS Minimum_Value,
        
        MAX("{column}") AS Maximum_Value,
        
        AVG("{column}") AS Average_Value
        
    FROM "{table}";
    """
    
    numerical_quality.append(
        pd.read_sql(query, engine)
    )

numerical_quality = pd.concat(
    numerical_quality,
    ignore_index=True
)

numerical_quality

,Table_Name,Column_Name,Total_Rows,Non_Null_Count,Null_Count,Zero_Count,Negative_Count,Minimum_Value,Maximum_Value,Average_Value
0,actors,actor_id,54588,54588,0,0,0,1,1893207,725024.789331
1,actors,gender,54588,54588,0,27725,0,0,2,0.792647
2,casts,movie_id,106257,106257,0,0,0,5,459488,56237.55713
3,casts,actor_id,106257,106257,0,0,0,1,1893207,453207.109565
4,genremap,movie_id,12160,12160,0,0,0,5,459488,50812.516694
5,genremap,genre_id,12160,12160,0,0,0,12,10770,1906.939638
6,genres,genre_id,20,20,0,0,0,12,10770,3758.7
7,keywordmap,movie_id,36194,36194,0,0,0,5,459488,40312.524148
8,keywordmap,keyword_id,36194,36194,0,0,0,30,238222,48872.688595
9,keywords,keyword_id,9813,9813,0,0,0,30,238222,93314.804953


In [46]:
numerical_quality["Null_Percentage"] = (
    numerical_quality["Null_Count"]
    / numerical_quality["Total_Rows"]
    * 100
)

numerical_quality["Zero_Percentage"] = (
    numerical_quality["Zero_Count"]
    / numerical_quality["Total_Rows"]
    * 100
)

numerical_quality["Negative_Percentage"] = (
    numerical_quality["Negative_Count"]
    / numerical_quality["Total_Rows"]
    * 100
)

numerical_quality

,Table_Name,Column_Name,Total_Rows,Non_Null_Count,Null_Count,Zero_Count,Negative_Count,Minimum_Value,Maximum_Value,Average_Value,Null_Percentage,Zero_Percentage,Negative_Percentage
0,actors,actor_id,54588,54588,0,0,0,1,1893207,725024.789331,0.0,0.0,0.0
1,actors,gender,54588,54588,0,27725,0,0,2,0.792647,0.0,50.789551,0.0
2,casts,movie_id,106257,106257,0,0,0,5,459488,56237.55713,0.0,0.0,0.0
3,casts,actor_id,106257,106257,0,0,0,1,1893207,453207.109565,0.0,0.0,0.0
4,genremap,movie_id,12160,12160,0,0,0,5,459488,50812.516694,0.0,0.0,0.0
5,genremap,genre_id,12160,12160,0,0,0,12,10770,1906.939638,0.0,0.0,0.0
6,genres,genre_id,20,20,0,0,0,12,10770,3758.7,0.0,0.0,0.0
7,keywordmap,movie_id,36194,36194,0,0,0,5,459488,40312.524148,0.0,0.0,0.0
8,keywordmap,keyword_id,36194,36194,0,0,0,30,238222,48872.688595,0.0,0.0,0.0
9,keywords,keyword_id,9813,9813,0,0,0,30,238222,93314.804953,0.0,0.0,0.0


## Date Validation

Date validation is performed to determine whether date-related fields contain complete, valid, and logically reasonable values.

The assessment will be performed across all tables and will examine:

- Date-related columns.
- NULL date values.
- Empty date values.
- Invalid date formats.
- Minimum and maximum dates.
- Dates outside the expected range.
- Future dates where they are not expected.

Because SQLite does not enforce a dedicated DATE data type, date values may be stored as TEXT, INTEGER, or other formats. Therefore, date validation must consider both the column name and the actual values stored in the column.

The results will help determine whether date fields can safely be used for temporal analysis and reporting.

In [47]:
from sqlalchemy import inspect

inspector = inspect(engine)

date_columns = []

for table in table_names:

    columns = inspector.get_columns(table)

    for column in columns:

        column_name = column["name"]
        data_type = str(column["type"]).upper()

        if (
            "DATE" in data_type
            or "TIME" in data_type
            or any(
                keyword in column_name.lower()
                for keyword in [
                    "date",
                    "time",
                    "year",
                    "created",
                    "updated",
                    "release"
                ]
            )
        ):

            date_columns.append({
                "Table_Name": table,
                "Column_Name": column_name,
                "Data_Type": data_type
            })

date_columns_df = pd.DataFrame(date_columns)

date_columns_df

,Table_Name,Column_Name,Data_Type
0,oscars,year,VARCHAR(10)
1,movies,release_date,DATETIME
2,movies,runtime,DOUBLE
3,movies,release_status,VARCHAR(50)


In [48]:
date_null_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Total_Rows,
        SUM(
            CASE
                WHEN "{column}" IS NULL
                THEN 1
                ELSE 0
            END
        ) AS Null_Count
    FROM "{table}";
    """

    result = pd.read_sql(query, engine)

    date_null_results.append(result)

date_null_profile = pd.concat(
    date_null_results,
    ignore_index=True
)

date_null_profile

,Table_Name,Column_Name,Total_Rows,Null_Count
0,oscars,year,9964,0
1,movies,release_date,4803,1
2,movies,runtime,4803,2
3,movies,release_status,4803,0


In [49]:
date_null_profile[
    date_null_profile["Null_Count"] > 0
]

,Table_Name,Column_Name,Total_Rows,Null_Count
1,movies,release_date,4803,1
2,movies,runtime,4803,2


In [50]:
date_empty_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Empty_Date_Count
    FROM "{table}"
    WHERE TRIM(CAST("{column}" AS TEXT)) = '';
    """

    result = pd.read_sql(query, engine)

    date_empty_results.append(result)

date_empty_profile = pd.concat(
    date_empty_results,
    ignore_index=True
)

date_empty_profile[
    date_empty_profile["Empty_Date_Count"] > 0
]

,Table_Name,Column_Name,Empty_Date_Count


In [51]:
date_range_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        MIN("{column}") AS Minimum_Date,
        MAX("{column}") AS Maximum_Date,
        COUNT("{column}") AS Valid_Value_Count
    FROM "{table}";
    """

    result = pd.read_sql(query, engine)

    date_range_results.append(result)

date_range_profile = pd.concat(
    date_range_results,
    ignore_index=True
)

date_range_profile

,Table_Name,Column_Name,Minimum_Date,Maximum_Date,Valid_Value_Count
0,oscars,year,1928,2015,9964
1,movies,release_date,1916-09-04 00:00:00.000000,2017-02-03 00:00:00.000000,4802
2,movies,runtime,0.0,338.0,4801
3,movies,release_status,Post Production,Rumored,4803


In [52]:
invalid_date_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Invalid_Date_Count
    FROM "{table}"
    WHERE "{column}" IS NOT NULL
      AND TRIM(CAST("{column}" AS TEXT)) <> ''
      AND date("{column}") IS NULL;
    """

    result = pd.read_sql(query, engine)

    invalid_date_results.append(result)

invalid_date_profile = pd.concat(
    invalid_date_results,
    ignore_index=True
)

invalid_date_profile[
    invalid_date_profile["Invalid_Date_Count"] > 0
]

,Table_Name,Column_Name,Invalid_Date_Count
3,movies,release_status,4803


In [53]:
future_date_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Future_Date_Count
    FROM "{table}"
    WHERE date("{column}") > date('now');
    """

    result = pd.read_sql(query, engine)

    future_date_results.append(result)

future_date_profile = pd.concat(
    future_date_results,
    ignore_index=True
)

future_date_profile[
    future_date_profile["Future_Date_Count"] > 0
]

,Table_Name,Column_Name,Future_Date_Count


In [54]:
old_date_results = []

for _, row in date_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Suspicious_Old_Date_Count
    FROM "{table}"
    WHERE date("{column}") < date('1800-01-01');
    """

    result = pd.read_sql(query, engine)

    old_date_results.append(result)

old_date_profile = pd.concat(
    old_date_results,
    ignore_index=True
)

old_date_profile[
    old_date_profile["Suspicious_Old_Date_Count"] > 0
]

,Table_Name,Column_Name,Suspicious_Old_Date_Count
0,oscars,year,9964
2,movies,runtime,4801


In [55]:
date_quality = (
    date_null_profile
    .merge(
        date_empty_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
    .merge(
        date_range_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
    .merge(
        invalid_date_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
    .merge(
        future_date_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
    .merge(
        old_date_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
)

In [56]:
count_columns = [
    "Null_Count",
    "Empty_Date_Count",
    "Invalid_Date_Count",
    "Future_Date_Count",
    "Suspicious_Old_Date_Count"
]

for column in count_columns:
    date_quality[column] = (
        date_quality[column]
        .fillna(0)
        .astype(int)
    )

date_quality

,Table_Name,Column_Name,Total_Rows,Null_Count,Empty_Date_Count,Minimum_Date,Maximum_Date,Valid_Value_Count,Invalid_Date_Count,Future_Date_Count,Suspicious_Old_Date_Count
0,oscars,year,9964,0,0,1928,2015,9964,0,0,9964
1,movies,release_date,4803,1,0,1916-09-04 00:00:00.000000,2017-02-03 00:00:00.000000,4802,0,0,0
2,movies,runtime,4803,2,0,0.0,338.0,4801,0,0,4801
3,movies,release_status,4803,0,0,Post Production,Rumored,4803,4803,0,0


In [57]:
date_quality["Status"] = date_quality.apply(
    lambda row: (
        "PASS"
        if (
            row["Null_Count"] == 0
            and row["Empty_Date_Count"] == 0
            and row["Invalid_Date_Count"] == 0
        )
        else "REVIEW"
    ),
    axis=1
)

date_quality

,Table_Name,Column_Name,Total_Rows,Null_Count,Empty_Date_Count,Minimum_Date,Maximum_Date,Valid_Value_Count,Invalid_Date_Count,Future_Date_Count,Suspicious_Old_Date_Count,Status
0,oscars,year,9964,0,0,1928,2015,9964,0,0,9964,PASS
1,movies,release_date,4803,1,0,1916-09-04 00:00:00.000000,2017-02-03 00:00:00.000000,4802,0,0,0,REVIEW
2,movies,runtime,4803,2,0,0.0,338.0,4801,0,0,4801,REVIEW
3,movies,release_status,4803,0,0,Post Production,Rumored,4803,4803,0,0,REVIEW


In [58]:
date_quality[
    date_quality["Status"] == "REVIEW"
]

,Table_Name,Column_Name,Total_Rows,Null_Count,Empty_Date_Count,Minimum_Date,Maximum_Date,Valid_Value_Count,Invalid_Date_Count,Future_Date_Count,Suspicious_Old_Date_Count,Status
1,movies,release_date,4803,1,0,1916-09-04 00:00:00.000000,2017-02-03 00:00:00.000000,4802,0,0,0,REVIEW
2,movies,runtime,4803,2,0,0.0,338.0,4801,0,0,4801,REVIEW
3,movies,release_status,4803,0,0,Post Production,Rumored,4803,4803,0,0,REVIEW


In [59]:
pd.read_sql("""
SELECT
    movie_id,
    title,
    release_date
FROM movies
WHERE release_date IS NOT NULL
  AND TRIM(release_date) <> ''
  AND date(release_date) IS NULL;
""", engine)

,movie_id,title,release_date


In [60]:
pd.read_sql("""
SELECT
    movie_id,
    title,
    release_date
FROM movies
WHERE date(release_date) > date('now')
ORDER BY release_date;
""", engine)

,movie_id,title,release_date


## Categorical Consistency Assessment

Categorical consistency evaluates whether categorical and identifier values are represented consistently across the database.

The assessment will examine:

- Distinct values within categorical columns.
- Frequency of categorical values.
- NULL and empty categorical values.
- Leading or trailing whitespace.
- Case inconsistencies.
- Unexpected categorical values.
- High-cardinality categorical fields.
- Identifier/code consistency.

The purpose is to identify values that may represent the same category but are stored differently.

Examples include:

- `Action` vs `action`
- `English` vs `english`
- `USA` vs ` USA`
- Different representations of the same status or code.

Categorical values will not be modified automatically. Identified inconsistencies will be investigated before any standardization is performed.

In [61]:
from sqlalchemy import inspect

inspector = inspect(engine)

categorical_columns = []

for table in table_names:

    columns = inspector.get_columns(table)

    for column in columns:

        column_name = column["name"]
        data_type = str(column["type"]).upper()

        if any(
            x in data_type
            for x in ["CHAR", "TEXT", "VARCHAR"]
        ):

            categorical_columns.append({
                "Table_Name": table,
                "Column_Name": column_name,
                "Data_Type": data_type
            })

categorical_columns_df = pd.DataFrame(categorical_columns)

categorical_columns_df

,Table_Name,Column_Name,Data_Type
0,actors,actor_name,VARCHAR(100)
1,casts,characters,VARCHAR(500)
2,oscars,year,VARCHAR(10)
3,oscars,award,VARCHAR(500)
4,oscars,winner,VARCHAR(10)
5,oscars,name,VARCHAR(500)
6,oscars,film,VARCHAR(500)
7,movies,title,VARCHAR(500)
8,movies,homepage,VARCHAR(500)
9,movies,original_language,VARCHAR(50)


In [62]:
categorical_cardinality = []

for _, row in categorical_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Total_Rows,
        COUNT(DISTINCT "{column}") AS Unique_Values,
        SUM(
            CASE
                WHEN "{column}" IS NULL
                THEN 1
                ELSE 0
            END
        ) AS Null_Count
    FROM "{table}";
    """

    result = pd.read_sql(query, engine)

    categorical_cardinality.append(result)

categorical_cardinality = pd.concat(
    categorical_cardinality,
    ignore_index=True
)

categorical_cardinality

,Table_Name,Column_Name,Total_Rows,Unique_Values,Null_Count
0,actors,actor_name,54588,54201,0
1,casts,characters,106257,71577,0
2,oscars,year,9964,88,0
3,oscars,award,9964,114,0
4,oscars,winner,9964,1,7643
5,oscars,name,9964,5747,0
6,oscars,film,9964,6363,333
7,movies,title,4803,4800,0
8,movies,homepage,4803,1691,3091
9,movies,original_language,4803,37,0


In [63]:
categorical_values = []

for _, row in categorical_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        "{column}" AS Category_Value,
        COUNT(*) AS Record_Count
    FROM "{table}"
    WHERE "{column}" IS NOT NULL
    GROUP BY "{column}"
    ORDER BY Record_Count DESC;
    """

    result = pd.read_sql(query, engine)

    categorical_values.append(result)

categorical_value_profile = pd.concat(
    categorical_values,
    ignore_index=True
)

categorical_value_profile

,Table_Name,Column_Name,Category_Value,Record_Count
0,actors,actor_name,Craig Johnson,4
1,actors,actor_name,Tom Quinn,3
2,actors,actor_name,Scott Walker,3
3,actors,actor_name,Robert Morgan,3
4,actors,actor_name,Richard Wilson,3
...,...,...,...,...
173419,productioncompanies,production_company_name,10 West Studios,1
173420,productioncompanies,production_company_name,1.85 Films,1
173421,productioncompanies,production_company_name,-frica Filmes,1
173422,productioncompanies,production_company_name,+sterreichischer Rundfunk (ORF),1


In [64]:
whitespace_results = []

for _, row in categorical_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        COUNT(*) AS Whitespace_Count
    FROM "{table}"
    WHERE "{column}" IS NOT NULL
      AND "{column}" != TRIM("{column}");
    """

    result = pd.read_sql(query, engine)

    whitespace_results.append(result)

whitespace_profile = pd.concat(
    whitespace_results,
    ignore_index=True
)

whitespace_profile[
    whitespace_profile["Whitespace_Count"] > 0
]

,Table_Name,Column_Name,Whitespace_Count
0,actors,actor_name,67
5,oscars,name,7309
11,movies,overview,1
14,keywords,keyword_name,10
21,productioncountries,production_country_name,1


In [65]:
case_variation_results = []

for _, row in categorical_columns_df.iterrows():

    table = row["Table_Name"]
    column = row["Column_Name"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        '{column}' AS Column_Name,
        LOWER(TRIM("{column}")) AS Normalized_Value,
        COUNT(DISTINCT "{column}") AS Original_Variations
    FROM "{table}"
    WHERE "{column}" IS NOT NULL
      AND TRIM("{column}") <> ''
    GROUP BY LOWER(TRIM("{column}"))
    HAVING COUNT(DISTINCT "{column}") > 1
    ORDER BY Original_Variations DESC;
    """

    result = pd.read_sql(query, engine)

    case_variation_results.append(result)

case_variation_profile = pd.concat(
    case_variation_results,
    ignore_index=True
)

case_variation_profile

,Table_Name,Column_Name,Normalized_Value,Original_Variations
0,actors,actor_name,shauna macdonald,2
1,actors,actor_name,norma argentina,2
2,actors,actor_name,nana gbewonyo,2
3,actors,actor_name,lo hoi-pang,2
4,actors,actor_name,chuck mitchell,2
...,...,...,...,...
176,productioncompanies,production_company_name,heyday films,2
177,productioncompanies,production_company_name,greenestreet films,2
178,productioncompanies,production_company_name,dreamworks pictures,2
179,productioncompanies,production_company_name,benderspink,2


In [66]:
categorical_cardinality[
    categorical_cardinality["Unique_Values"] > 100
].sort_values(
    "Unique_Values",
    ascending=False
)

,Table_Name,Column_Name,Total_Rows,Unique_Values,Null_Count
1,casts,characters,106257,71577,0
0,actors,actor_name,54588,54201,0
14,keywords,keyword_name,9813,9811,0
6,oscars,film,9964,6363,333
5,oscars,name,9964,5747,0
22,productioncompanies,production_company_name,5047,5017,0
7,movies,title,4803,4800,0
11,movies,overview,4803,4800,3
10,movies,original_title,4803,4750,0
13,movies,tagline,4803,3944,844


In [67]:
categorical_quality = (
    categorical_cardinality
    .merge(
        whitespace_profile,
        on=["Table_Name", "Column_Name"],
        how="left"
    )
)

categorical_quality["Whitespace_Count"] = (
    categorical_quality["Whitespace_Count"]
    .fillna(0)
    .astype(int)
)

categorical_quality

,Table_Name,Column_Name,Total_Rows,Unique_Values,Null_Count,Whitespace_Count
0,actors,actor_name,54588,54201,0,67
1,casts,characters,106257,71577,0,0
2,oscars,year,9964,88,0,0
3,oscars,award,9964,114,0,0
4,oscars,winner,9964,1,7643,0
5,oscars,name,9964,5747,0,7309
6,oscars,film,9964,6363,333,0
7,movies,title,4803,4800,0,0
8,movies,homepage,4803,1691,3091,0
9,movies,original_language,4803,37,0,0


## Many-to-Many Relationship Validation

The movie database contains several many-to-many relationships.

A movie may be associated with:

- Multiple actors.
- Multiple genres.
- Multiple keywords.
- Multiple languages.
- Multiple production companies.
- Multiple production countries.

These relationships are implemented through bridge tables.

The objective of this assessment is to validate:

1. The grain of each bridge table.
2. Duplicate relationship mappings.
3. Number of related entities per parent.
4. Number of parents per related entity.
5. Referential integrity.
6. Unexpected relationship cardinality.

Multiple records for the same movie are not automatically duplicates. A movie having multiple genres, languages, companies, or countries is expected behavior.

A duplicate relationship occurs when the same pair of entities appears more than once.

In [68]:
many_to_many_relationships = {
    "casts": {
        "left": "movie_id",
        "right": "actor_id"
    },

    "genremap": {
        "left": "movie_id",
        "right": "genre_id"
    },

    "keywordmap": {
        "left": "movie_id",
        "right": "keyword_id"
    },

    "languagemap": {
        "left": "movie_id",
        "right": "ISO_639_1"
    },

    "productioncompanymap": {
        "left": "movie_id",
        "right": "production_company_id"
    },

    "productioncountrymap": {
        "left": "movie_id",
        "right": "ISO_3166_1"
    }
}

In [69]:
relationship_duplicates = []

for table, relationship in many_to_many_relationships.items():

    left_key = relationship["left"]
    right_key = relationship["right"]

    query = f"""
    SELECT
        '{table}' AS Table_Name,
        {left_key},
        {right_key},
        COUNT(*) AS Record_Count
    FROM {table}
    GROUP BY
        {left_key},
        {right_key}
    HAVING COUNT(*) > 1
    ORDER BY Record_Count DESC;
    """

    result = pd.read_sql(query, engine)

    relationship_duplicates.append(result)

relationship_duplicate_profile = pd.concat(
    relationship_duplicates,
    ignore_index=True
)

relationship_duplicate_profile

,Table_Name,movie_id,actor_id,Record_Count,genre_id,keyword_id,iso_639_1,production_company_id,iso_3166_1
0,casts,182291,567620.0,5,NaN,NaN,NaN,NaN,NaN
1,casts,18360,85757.0,3,NaN,NaN,NaN,NaN,NaN
2,casts,18360,130565.0,3,NaN,NaN,NaN,NaN,NaN
3,casts,18360,130566.0,3,NaN,NaN,NaN,NaN,NaN
4,casts,38167,1456290.0,3,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
160,casts,329833,85757.0,2,NaN,NaN,NaN,NaN,NaN
161,casts,333355,31162.0,2,NaN,NaN,NaN,NaN,NaN
162,casts,333355,39214.0,2,NaN,NaN,NaN,NaN,NaN
163,casts,339984,156100.0,2,NaN,NaN,NaN,NaN,NaN


In [70]:
movie_relationship_counts = []

for table, relationship in many_to_many_relationships.items():

    left_key = relationship["left"]
    right_key = relationship["right"]

    query = f"""
    SELECT
        '{table}' AS Relationship_Table,
        {left_key} AS Movie_ID,
        COUNT(DISTINCT {right_key}) AS Related_Entity_Count
    FROM {table}
    GROUP BY {left_key}
    ORDER BY Related_Entity_Count DESC;
    """

    result = pd.read_sql(query, engine)

    movie_relationship_counts.append(result)

movie_relationship_profile = pd.concat(
    movie_relationship_counts,
    ignore_index=True
)

movie_relationship_profile

,Relationship_Table,Movie_ID,Related_Entity_Count
0,casts,80585,224
1,casts,3083,213
2,casts,82695,208
3,casts,324668,208
4,casts,10661,183
...,...,...,...
27719,productioncountrymap,407887,1
27720,productioncountrymap,417859,1
27721,productioncountrymap,433715,1
27722,productioncountrymap,447027,1


In [71]:
relationship_summary = []

for table, relationship in many_to_many_relationships.items():

    left_key = relationship["left"]
    right_key = relationship["right"]

    query = f"""
    SELECT
        '{table}' AS Relationship_Table,
        COUNT(*) AS Total_Mappings,
        COUNT(DISTINCT {left_key}) AS Unique_Movies,
        COUNT(DISTINCT {right_key}) AS Unique_Related_Entities,
        ROUND(
            CAST(COUNT(*) AS FLOAT)
            / NULLIF(COUNT(DISTINCT {left_key}), 0),
            2
        ) AS Avg_Relationships_Per_Movie
    FROM {table};
    """

    result = pd.read_sql(query, engine)

    relationship_summary.append(result)

relationship_summary = pd.concat(
    relationship_summary,
    ignore_index=True
)

relationship_summary

,Relationship_Table,Total_Mappings,Unique_Movies,Unique_Related_Entities,Avg_Relationships_Per_Movie
0,casts,106257,4760,54588,22.32
1,genremap,12160,4775,20,2.55
2,keywordmap,36194,4391,9813,8.24
3,languagemap,6937,4717,87,1.47
4,productioncompanymap,13677,4452,5047,3.07
5,productioncountrymap,6436,4629,88,1.39


In [72]:
pd.read_sql("""
SELECT
    actor_id,
    COUNT(DISTINCT movie_id) AS Movie_Count
FROM casts
GROUP BY actor_id
ORDER BY Movie_Count DESC;
""", engine).head(20)

,actor_id,Movie_Count
0,2231,67
1,380,57
2,62,51
3,1892,48
4,192,46
5,884,43
6,3896,41
7,85,40
8,887,40
9,1230,39


In [73]:
reverse_relationship_counts = []

for table, relationship in many_to_many_relationships.items():

    left_key = relationship["left"]
    right_key = relationship["right"]

    query = f"""
    SELECT
        '{table}' AS Relationship_Table,
        {right_key} AS Related_Entity_ID,
        COUNT(DISTINCT {left_key}) AS Parent_Count
    FROM {table}
    GROUP BY {right_key}
    ORDER BY Parent_Count DESC;
    """

    result = pd.read_sql(query, engine)

    reverse_relationship_counts.append(result)

reverse_relationship_profile = pd.concat(
    reverse_relationship_counts,
    ignore_index=True
)

reverse_relationship_profile

,Relationship_Table,Related_Entity_ID,Parent_Count
0,casts,2231,67
1,casts,380,57
2,casts,62,51
3,casts,1892,48
4,casts,192,46
...,...,...,...
69638,productioncountrymap,PT,1
69639,productioncountrymap,RS,1
69640,productioncountrymap,TN,1
69641,productioncountrymap,TR,1


In [74]:
movie_relationship_profile[
    movie_relationship_profile["Related_Entity_Count"] > 50
].sort_values(
    "Related_Entity_Count",
    ascending=False
)

,Relationship_Table,Movie_ID,Related_Entity_Count
0,casts,80585,224
1,casts,3083,213
2,casts,82695,208
3,casts,324668,208
4,casts,10661,183
...,...,...,...
327,casts,2501,51
326,casts,1966,51
325,casts,745,51
324,casts,673,51


In [75]:
many_to_many_quality = relationship_summary.copy()

many_to_many_quality["Duplicate_Mappings"] = (
    many_to_many_quality["Relationship_Table"]
    .map(
        relationship_duplicate_profile
        .groupby("Table_Name")
        .size()
    )
    .fillna(0)
    .astype(int)
)

many_to_many_quality

,Relationship_Table,Total_Mappings,Unique_Movies,Unique_Related_Entities,Avg_Relationships_Per_Movie,Duplicate_Mappings
0,casts,106257,4760,54588,22.32,165
1,genremap,12160,4775,20,2.55,0
2,keywordmap,36194,4391,9813,8.24,0
3,languagemap,6937,4717,87,1.47,0
4,productioncompanymap,13677,4452,5047,3.07,0
5,productioncountrymap,6436,4629,88,1.39,0


In [76]:
many_to_many_quality["Status"] = many_to_many_quality.apply(
    lambda row: (
        "PASS"
        if row["Duplicate_Mappings"] == 0
        else "REVIEW"
    ),
    axis=1
)

many_to_many_quality

,Relationship_Table,Total_Mappings,Unique_Movies,Unique_Related_Entities,Avg_Relationships_Per_Movie,Duplicate_Mappings,Status
0,casts,106257,4760,54588,22.32,165,REVIEW
1,genremap,12160,4775,20,2.55,0,PASS
2,keywordmap,36194,4391,9813,8.24,0,PASS
3,languagemap,6937,4717,87,1.47,0,PASS
4,productioncompanymap,13677,4452,5047,3.07,0,PASS
5,productioncountrymap,6436,4629,88,1.39,0,PASS


## Data Quality Summary

The results from the individual validation checks will be consolidated into a data quality summary.

Each identified issue should be classified according to:

- Data quality dimension
- Table
- Column
- Issue
- Number of affected records
- Severity
- Recommended action

# Data Quality Findings

## Key Findings

The data quality assessment identified the following observations:

1. **Completeness:** ...
2. **Uniqueness:** ...
3. **Referential Integrity:** ...
4. **Validity:** ...
5. **Consistency:** ...
6. **Many-to-Many Relationships:** ...

## Impact on Analytics

The identified issues were assessed based on their potential impact on downstream analytics.

Particular attention should be given to referential integrity and many-to-many relationships because incorrect handling of these relationships can result in missing records or unintended row multiplication during analytical joins.

## Recommended Actions

Based on the assessment:

- Preserve valid many-to-many relationships.
- Investigate orphan records before building analytics marts.
- Document fields with legitimate missing values.
- Treat zero financial values carefully.
- Avoid using blanket replacements without understanding the business meaning of missing data.
- Validate the grain of each analytics mart after transformation.

## Conclusion

The data quality assessment provides a baseline for determining whether the source database is suitable for analytical transformation.

The next stage is **SQL Joins**, where the validated relationships will be used to demonstrate how the normalized database can be queried and transformed into analysis-ready datasets.